<a href="https://colab.research.google.com/github/PatrickJReed/juggle-classifier/blob/main/notebooks/03_predict_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03 — Predict and evaluate

Runs the trained classifier on a video's features, then evaluates against labels (if present).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
REPO_URL = 'https://github.com/<your-user>/juggle-classifier.git'  # EDIT
REPO_DIR = '/content/juggle-classifier'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git pull
!pip install -q -r requirements-colab.txt


In [ ]:
VIDEO = 'Juggles_New'  # EDIT — predict + evaluate against this video
MODEL = 'models/juggle_classifier.pkl'
THRESHOLD = 0.5
NMS_WINDOW = 5
!python -m tools.predict --features features/{VIDEO}.csv --model {MODEL} \
    --output events/{VIDEO}.csv --threshold {THRESHOLD} --nms-window {NMS_WINDOW}
import os
if os.path.exists(f'labels/{VIDEO}.csv'):
    !python -m tools.evaluate --pred events/{VIDEO}.csv --gt labels/{VIDEO}.csv --tolerance 5
else:
    print(f'No labels for {VIDEO}; printing total only.')
    import pandas as pd
    print(pd.read_csv(f'events/{VIDEO}.csv').groupby('foot').size())


In [ ]:
# Optional: sweep threshold/window to find best F1 on validation
import subprocess, sys
VAL_VIDEO = 'Clark_Juggles1'  # EDIT — held-out validation video stem
for thr in [0.3, 0.4, 0.5, 0.6]:
    for w in [3, 5, 7]:
        print(f'\n--- threshold={thr} nms_window={w} ---')
        subprocess.run([sys.executable, '-m', 'tools.predict',
                        '--features', f'features/{VAL_VIDEO}.csv',
                        '--model', 'models/juggle_classifier.pkl',
                        '--output', f'events/{VAL_VIDEO}.csv',
                        '--threshold', str(thr), '--nms-window', str(w)], check=True)
        subprocess.run([sys.executable, '-m', 'tools.evaluate',
                        '--pred', f'events/{VAL_VIDEO}.csv',
                        '--gt', f'labels/{VAL_VIDEO}.csv',
                        '--tolerance', '5'], check=True)
